# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/dasinithin/flyrank-ml-internship-starter/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

## 1. Unit of analysis + time window

**Unit of Analysis:** One row = one pseudonymized content piece (`content_id`) belonging to a specific client domain (`client_id`).

**Time Window:** Aggregated over a 90-day observation window ending on the export snapshot date. All features represent retrospective search and engagement behavior strictly prior to the forecast decision point.

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv('data/raw/content_refresh_anonymized.csv')
print(f"Total verified rows: {len(df):,}")
print(f"Unique content IDs: {df['content_id'].nunique():,}")
print(f"Unique client IDs: {df['client_id'].nunique():,}")


Total verified rows: 30,000
Unique content IDs: 30,000
Unique client IDs: 32


## 2. Fields: feature / label / context / excluded

We categorize all 44 fields into four strict buckets:
1. **Features (Model Inputs):**
   - *Numerical (18):* `search_volume`, `competition`, `cpc`, `word_count`, `char_count`, `log_impressions_90d`, `log_clicks_90d`, `log_sessions_90d`, `log_ai_sessions_90d`, `days_with_impressions`, `days_with_sessions`, `content_age_days`, `days_since_last_update`, `ctr`, `avg_position`, `engagement_rate`, `scroll_rate`, `ai_traffic_pct`.
   - *Categorical (8):* `competition_level`, `content_type`, `main_intent`, `age_tier`, `freshness_tier`, `word_count_tier`, `impression_tier`, `position_tier`.
2. **Label:** `is_declining_label` (derived from `trend_direction == 'down'`).
3. **Context / Split Keys:** `content_id` (identifier), `client_id` (grouping key for client-holdout split).
4. **Excluded Fields:** `trend_direction` and `trend_pct` (direct target leakage), raw client identifiers (memorization risk).

In [2]:
features_num = [
    'search_volume', 'competition', 'cpc', 'word_count', 'char_count',
    'impressions_90d', 'clicks_90d', 'sessions_90d', 'ai_sessions_90d',
    'days_with_impressions', 'days_with_sessions', 'content_age_days',
    'days_since_last_update', 'ctr', 'avg_position', 'engagement_rate',
    'scroll_rate', 'ai_traffic_pct'
]
features_cat = [
    'competition_level', 'content_type', 'main_intent', 'age_tier',
    'freshness_tier', 'word_count_tier', 'impression_tier', 'position_tier'
]
excluded = ['trend_direction', 'trend_pct', 'client_id', 'content_id']

print(f"Features: {len(features_num)} numeric + {len(features_cat)} categorical = {len(features_num) + len(features_cat)} total")
print(f"Excluded: {excluded}")


Features: 18 numeric + 8 categorical = 26 total
Excluded: ['trend_direction', 'trend_pct', 'client_id', 'content_id']


## 3. Verify it with queries (grain, counts, missing values, windows)

We run verification checks confirming data health, missingness handling, and floor criteria.

In [3]:
# Check missing values
missing = df[features_num + features_cat].isnull().sum()
missing_cols = missing[missing > 0]
print(f"Columns with missing values: {len(missing_cols)}")
if len(missing_cols) > 0:
    print(missing_cols)
else:
    print("Zero unhandled null values in feature columns.")

# Check minimum floors
print(f"Min impressions_90d: {df['impressions_90d'].min()}")
print(f"Min content_age_days: {df['content_age_days'].min()}")


Columns with missing values: 9
search_volume        2468
competition          2468
cpc                  2468
word_count           7699
char_count           7699
scroll_rate           125
competition_level    2610
main_intent          2374
word_count_tier      7699
dtype: int64
Min impressions_90d: 1
Min content_age_days: 90


## 4. Data limits

**Observational Constraints:**
1. **GSC Aggregation Thresholds:** Long-tail queries with very low click volumes are aggregated or omitted by Search Console privacy filters.
2. **Unbalanced Tracking:** Certain clients have shorter GA4 history than GSC history, requiring `ga4_data_available` verification.
3. **Non-Causal Nature:** Features record correlation with decline, but do not prove that altering any single feature (such as increasing word count) will unilaterally reverse decay.

In [4]:
# Check density of analytics signals
print(f"Rows with impressions: {(df['impressions_90d'] > 0).mean():.1%}")
print(f"Rows with clicks: {(df['clicks_90d'] > 0).mean():.1%}")
print(f"Rows with GA4 sessions: {(df['sessions_90d'] > 0).mean():.1%}")
print(f"Rows with AI-referral sessions: {(df['ai_sessions_90d'] > 0).mean():.1%}")


Rows with impressions: 100.0%
Rows with clicks: 56.0%
Rows with GA4 sessions: 100.0%
Rows with AI-referral sessions: 6.4%


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.